In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict, Literal
from pydantic import BaseModel, Field

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGINGFACEHUB_API_TOKEN")

In [4]:
model=HuggingFaceEndpoint(repo_id='mistralai/Mistral-7B-Instruct-v0.2',
    temperature=0.1,
    max_new_tokens=2048,
    task='conversational',
    huggingfacehub_api_token=secret_value_0)

In [5]:
llm=ChatHuggingFace(llm=model)

In [22]:
class Sentimentu(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review whether its positive or negative')

In [23]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [24]:
class Review(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [25]:
from langchain_core.output_parsers import PydanticOutputParser

In [26]:
prs1=PydanticOutputParser(pydantic_object=Sentimentu)
prs2=PydanticOutputParser(pydantic_object=DiagnosisSchema)

In [27]:
from langchain_core.prompts import PromptTemplate

In [28]:
def find_sentiment(state: Review):
    tem=PromptTemplate(template='''For the following review find out the sentiment \n {inn} \n{par}''',
                       input_variabeles={'inn':state["review"]},
                   partial_variables={'par':prs1.get_format_instructions()})
    strt=tem|llm|prs1
    sentiment = strt.invoke(tem).sentiment
    return {'sentiment': sentiment}

In [29]:
def check_sentiment(state: Review) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [30]:
def positive_response(state: Review):
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    response = llm.invoke(prompt).content
    return {'response': response}

In [31]:
def run_diagnosis(state: Review):
    tem=PromptTemplate(template='''Diagnose this negative review:\n\n{inn}\n"
    "Return issue_type, tone, and urgency. \n{par}''',
                input_variables={'inn':state['review']},
                   partial_variables={'par':prs2.get_format_instructions()})
    strt=tem|llm|prs2
    response = strt.invoke(tem)
    return {'diagnosis': response.model_dump()}

In [32]:
def negative_response(state: Review):
    diagnosis = state['diagnosis']
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = llm.invoke(prompt).content
    return {'response': response}

In [17]:
graph = StateGraph(Review)

In [18]:
graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

In [19]:
graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

In [33]:
a=graph.compile()

In [ ]:
a.invoke({'review':''})